In [ ]:

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

# ==========================================
# 1. Definir os caminhos dos arquivos
# ==========================================

arquivo_microdados = Path(
    "01_PENSE2024_MICRODADOS.csv"
)

arquivo_recomendadas = Path(
    "01_Colunas_recomendadas_pense_2024_final.xlsx"
)

arquivo_saida = Path(
    "02_PENSE2024_MICRODADOS_FILTRADO.parquet"
)

# ==========================================
# 2. Ler a planilha de colunas recomendadas
# ==========================================

df_recomendadas = pd.read_excel(
    arquivo_recomendadas
)

codigos_recomendados = set(
    df_recomendadas["Código da coluna"]
    .dropna()
    .astype(str)
    .str.strip()
)

# ==========================================
# 3. Identificar as colunas do CSV
# ==========================================

colunas_microdados = pd.read_csv(
    arquivo_microdados,
    sep=";",
    encoding="utf-8-sig",
    nrows=0
).columns.tolist()

colunas_mantidas = [
    coluna
    for coluna in colunas_microdados
    if coluna.strip() in codigos_recomendados
]

codigos_ausentes = (
    codigos_recomendados - set(colunas_microdados)
)

# ==========================================
# 4. Processar e gravar em Parquet
# ==========================================

escritor = None
total_linhas = 0

try:

    for bloco in pd.read_csv(
        arquivo_microdados,
        sep=";",
        encoding="utf-8-sig",
        usecols=colunas_mantidas,
        chunksize=10000,
        low_memory=False
    ):

        # Converter o bloco para tabela Arrow
        tabela = pa.Table.from_pandas(
            bloco,
            preserve_index=False
        )

        # Criar o escritor no primeiro bloco
        if escritor is None:

            escritor = pq.ParquetWriter(
                arquivo_saida,
                tabela.schema,
                compression="snappy"
            )

        # Gravar o bloco no arquivo Parquet
        escritor.write_table(tabela)

        total_linhas += len(bloco)

        print(
            f"Linhas processadas: {total_linhas:,}"
        )

finally:

    if escritor is not None:
        escritor.close()

# ==========================================
# 5. Relatório final
# ==========================================

print("\n" + "=" * 50)
print("PROCESSAMENTO CONCLUÍDO")
print("=" * 50)

print(
    f"Total de linhas: {total_linhas:,}"
)

print(
    f"Colunas originais: {len(colunas_microdados)}"
)

print(
    f"Colunas mantidas: {len(colunas_mantidas)}"
)

print(
    f"Colunas removidas: "
    f"{len(colunas_microdados) - len(colunas_mantidas)}"
)

print(
    f"Códigos recomendados não encontrados: "
    f"{len(codigos_ausentes)}"
)

print(f"Arquivo gerado: {arquivo_saida}")

print(
    f"Tamanho do arquivo: "
    f"{arquivo_saida.stat().st_size / (1024**2):.2f} MB"
)

print("=" * 50)
